In [1]:
%pip install pandas numpy faker


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
import random

# Initialize Faker and Seed for reproducibility
fake = Faker()
np.random.seed(42)

# 1. Define Business Parameters
NUM_RECORDS = 50000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2026, 5, 15)

locations = ['Sydney CBD', 'Melbourne Chadstone', 'Brisbane Queen St', 'Perth Hay St', 'Online Flagship']
# Weighting locations: Online and Sydney get more traffic
location_weights = [0.25, 0.20, 0.15, 0.10, 0.30] 

categories = ['Engagement Rings', 'Luxury Watches', 'Necklaces & Pendants', 'Earrings', 'Bracelets']
# Weighting categories: Everyday items sell more frequently than engagement rings
category_weights = [0.10, 0.15, 0.30, 0.25, 0.20]

print("Generating luxury retail data. This usually takes 5-10 seconds...")

# 2. Generate Base Data (Dates, Locations, Categories)
dates = [START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days)) for _ in range(NUM_RECORDS)]
dates.sort() # Sort chronologically

df = pd.DataFrame({
    'TransactionID': [f"TXN-{100000 + i}" for i in range(NUM_RECORDS)],
    'Date': dates,
    'StoreLocation': np.random.choice(locations, NUM_RECORDS, p=location_weights),
    'ProductCategory': np.random.choice(categories, NUM_RECORDS, p=category_weights),
    'Quantity': np.random.choice([1, 2, 3], NUM_RECORDS, p=[0.85, 0.10, 0.05]) # Luxury is usually low quantity per txn
})

# 3. Apply "Intelligent" Business Logic for Pricing
def assign_price(category):
    if category == 'Engagement Rings':
        return round(np.random.uniform(5000, 35000), 2)
    elif category == 'Luxury Watches':
        return round(np.random.uniform(3000, 25000), 2)
    elif category == 'Necklaces & Pendants':
        return round(np.random.uniform(500, 4500), 2)
    elif category == 'Earrings':
        return round(np.random.uniform(300, 2500), 2)
    else: # Bracelets
        return round(np.random.uniform(400, 3000), 2)

df['UnitValue'] = df['ProductCategory'].apply(assign_price)

# Sydney stores have a slight premium/upsell trend (+10% on average)
df.loc[df['StoreLocation'] == 'Sydney CBD', 'UnitValue'] *= 1.10
df['UnitValue'] = df['UnitValue'].round(2)
df['TotalRevenue'] = df['UnitValue'] * df['Quantity']

# 4. Generate Operational Metrics (Delivery & Inventory)
# Online orders have delivery times; in-store is usually immediate (0 days) unless it's a custom ring
df['DeliveryDelayDays'] = np.where(
    df['StoreLocation'] == 'Online Flagship',
    np.random.poisson(lam=3, size=NUM_RECORDS), # Average 3 days for online
    np.where(df['ProductCategory'] == 'Engagement Rings', np.random.poisson(lam=14, size=NUM_RECORDS), 0) # Custom rings take 14 days
)

# 5. Apply "Intelligent" Business Logic for Customer Reviews/Sentiment
# Base rating is 4 or 5
df['CustomerRating'] = np.random.choice([3, 4, 5], NUM_RECORDS, p=[0.1, 0.4, 0.5])

# Penalize rating heavily if delivery is delayed (> 5 days for standard, > 21 days for custom)
df.loc[(df['StoreLocation'] == 'Online Flagship') & (df['DeliveryDelayDays'] > 5), 'CustomerRating'] -= np.random.randint(1, 3)
df.loc[(df['ProductCategory'] == 'Engagement Rings') & (df['DeliveryDelayDays'] > 21), 'CustomerRating'] -= np.random.randint(1, 4)

# Ensure ratings stay between 1 and 5
df['CustomerRating'] = df['CustomerRating'].clip(1, 5)

# Generate basic synthetic sentiment categories based on rating
conditions = [
    (df['CustomerRating'] >= 4),
    (df['CustomerRating'] == 3),
    (df['CustomerRating'] <= 2)
]
choices = ['Positive', 'Neutral', 'Negative']
df['SentimentLabel'] = np.select(conditions, choices, default='Neutral')

# 6. Final Clean and Export
df['Date'] = pd.to_datetime(df['Date']).dt.date

print(f"Data generation complete. Total Rows: {df.shape[0]}")
print("\nFirst 5 rows:")
display(df.head())

# Export to CSV directly to your Mac Desktop to avoid permissions issues!
df.to_csv("~/Desktop/jewellery_retail_data.csv", index=False)
print("\n✅ Success! Check your Mac Desktop for the 'jewellery_retail_data.csv' file.")

Generating luxury retail data. This usually takes 5-10 seconds...
Data generation complete. Total Rows: 50000

First 5 rows:


,TransactionID,Date,StoreLocation,ProductCategory,Quantity,UnitValue,TotalRevenue,DeliveryDelayDays,CustomerRating,SentimentLabel
0,TXN-100000,2024-01-01,Melbourne Chadstone,Bracelets,1,579.95,579.95,0,3,Neutral
1,TXN-100001,2024-01-01,Online Flagship,Necklaces & Pendants,1,2842.79,2842.79,2,5,Positive
2,TXN-100002,2024-01-01,Online Flagship,Luxury Watches,1,20575.11,20575.11,3,5,Positive
3,TXN-100003,2024-01-01,Brisbane Queen St,Earrings,1,1981.84,1981.84,0,4,Positive
4,TXN-100004,2024-01-01,Sydney CBD,Necklaces & Pendants,1,4234.54,4234.54,0,4,Positive



✅ Success! Check your Mac Desktop for the 'jewellery_retail_data.csv' file.
